# 01. Statcast 원본 데이터 수집

2024 · 2025 정규시즌 투구 데이터를 월 단위 parquet 으로 Google Drive 에 저장한다.

- **월별로 나눠 저장** → 세션이 끊겨도 이어받기 가능 (이미 받은 달은 건너뜀)
- `game_type == 'R'` 로 **정규시즌만** 남김 (시범경기·포스트시즌 제외)
- 학습: 2024 시즌 / 테스트: 2025 시즌 (시즌 단위 분할로 시간 누수 차단)

In [ ]:
!pip install -q pybaseball

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
RAW = Path('/content/drive/MyDrive/baseball_ai/data/raw')
RAW.mkdir(parents=True, exist_ok=True)
print(RAW)

In [ ]:
import calendar, warnings
warnings.filterwarnings('ignore')

import pandas as pd
from pybaseball import statcast


def months(start, end):
    """'YYYY-MM' 범위를 (연, 월) 목록으로 펼친다."""
    sy, sm = map(int, start.split('-'))
    ey, em = map(int, end.split('-'))
    y, m = sy, sm
    while (y, m) <= (ey, em):
        yield y, m
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)


def fetch(year, month):
    out = RAW / f'statcast_{year}_{month:02d}.parquet'
    if out.exists():
        print(f'[건너뜀] {out.name}')
        return out

    last = calendar.monthrange(year, month)[1]
    start, end = f'{year}-{month:02d}-01', f'{year}-{month:02d}-{last}'
    print(f'[받는 중] {start} ~ {end}', flush=True)

    df = statcast(start_dt=start, end_dt=end)
    if df is None or df.empty:
        print('  -> 데이터 없음 (비시즌)')
        return None

    # 정규시즌만 남긴다. R=정규, S=시범, F/D/L/W=포스트시즌
    df = df[df['game_type'] == 'R']
    if df.empty:
        print('  -> 정규시즌 경기 없음')
        return None

    df.to_parquet(out, index=False)
    print(f'  -> {len(df):,}구 저장 ({out.stat().st_size/1e6:.1f}MB)')
    return out

## 다운로드 실행

한 달에 2~5분쯤 걸린다. 세션이 끊기면 이 셀만 다시 실행하면 된다 (받은 달은 건너뜀).

In [ ]:
for y, m in months('2024-03', '2024-10'):
    fetch(y, m)

In [ ]:
for y, m in months('2025-03', '2025-10'):
    fetch(y, m)

## 수집 결과 확인

In [ ]:
import pandas as pd

files = sorted(RAW.glob('statcast_*.parquet'))
rows = []
for f in files:
    df = pd.read_parquet(f, columns=['game_date', 'game_pk'])
    rows.append({
        '파일': f.name,
        '투구수': len(df),
        '시작': df['game_date'].min(),
        '종료': df['game_date'].max(),
        'MB': round(f.stat().st_size / 1e6, 1),
    })

summary = pd.DataFrame(rows)
display(summary)
print(f"
합계 {summary['투구수'].sum():,}구 / {summary['MB'].sum():.0f}MB")